[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# DataFrame &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


**1.** Make a table from `LATITUDES` with two columns, `station` and `latitude`, and print its shape and dtypes.

In [2]:
latitudes = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print(latitudes.shape)
print(latitudes.dtypes)

(5, 2)
station      object
latitude    float64
dtype: object


The keys and the values of the dictionary become two lists of the same length, five each, so the table has five rows, Kirkenes among them. This is how Setup builds `stations`.

**2.** Build a table from the first 12 tuples of `year_of_readings()`, and print its dtypes. Say why `time` is `object`.

In [3]:
rows = list(year_of_readings())[:12]
first_hours = pd.DataFrame(rows, columns=["station", "time", "celsius"])

print(first_hours.dtypes)

station     object
time        object
celsius    float64
dtype: object


`year_of_readings` yields each time as text, such as `2025-01-01T00:00`, and pandas keeps text as `object`. Setup converts the column with `pd.to_datetime`.

**3.** Make the wide table of the readings, with a column for each station and the hours as the index, and print the four readings at 03:00 on 12 January.

In [4]:
values = readings["celsius"].to_numpy().reshape(8760, 4)
wide = pd.DataFrame(values, columns=list(STATIONS), index=readings["time"].unique())

print(wide.loc["2025-01-12 03:00"])

Bergen      -4.6
Oslo        -6.2
Svalbard   -17.3
Tromso      -7.7
Name: 2025-01-12 03:00:00, dtype: float64


Svalbard's -17.3 is its coldest reading of the year, as the **Why Pandas** notebook found, and the other three stations read between -4.6 and -7.7 at the same hour.

**4.** Print how many rows and how many columns `readings` has, and its column names, without printing the table.

In [5]:
row_count, column_count = readings.shape

print(row_count, "rows,", column_count, "columns")
print(readings.columns.tolist())

35040 rows, 3 columns
['station', 'time', 'celsius']


`shape` is a tuple, so it unpacks into two names.

**5.** Build a table from three records, one of which has no `latitude`, and print it. Say what pandas put in the missing place, and what type the column became.

In [6]:
records = [
    {"station": "Bergen", "latitude": 60.39},
    {"station": "Kirkenes"},
    {"station": "Tromso", "latitude": 69.65},
]
described = pd.DataFrame(records)

print(described)
print(described.dtypes)

    station  latitude
0    Bergen     60.39
1  Kirkenes       NaN
2    Tromso     69.65
station      object
latitude    float64
dtype: object


The Kirkenes record has no `latitude`, so pandas put `NaN` there. The other values are decimals and `NaN` is a decimal too, so the column stays `float64`.

---

&#8592; **Back to:** [DataFrame](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/03-dataframe.ipynb)  &nbsp;&middot;&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)
